# Free-Droid (Szabi) — v4 fine-tune (tool-bővített dataset + új SYSTEM prompt)

Vékony futtató: telepíti az Unsloth-ot, klónozza a repót, és a `training/finetune.py`-t hívja.
Minden logika a `finetune.py` + `config.py`-ban (verziókövetett).

**Mi új a v3-hoz képest:**
- A v3 benchmark (2026-07-01) elérte a célját (koherencia/provokáció visszajött), **de a tool_calling
  minden méretnél gyenge maradt (1–2/5)** — megerősített dataset-hiány, nem hiperparaméter.
- **Dataset 626 → 681** (train 613 / val 68): +35 tool-példa (22 **összetett/láncolt** parancs +
  13 **kétértelmű** → `request_navigation_help` vagy őszinte visszakérdezés tool nélkül) + ~20 persona
  köszönés (`freedroid-ext.json`). Tool-arány **20,8% → 23,5%**, összetett hívás **7 → 29**.
- **Új SYSTEM prompt:** a `finetune.py` a `system_prompt.txt`-ből tanul — a v4 mostantól a
  **fizikai képességekkel bővített** prompttal tanul (lánctalp, előre/hátra, balra/jobbra, mozgó
  kamera-fej, wifi-rálátás CLI-vel). A v3 GGUF-ok még a régi, tréningkori promptot vitték.
- **Recept változatlan:** `--preset gentle` (lr 5e-5, 1 epoch, r/alpha 8) — a tool-gap dataset-ügy,
  nem receptügy; a gentle marad.

**Először: Runtime → Change runtime type → T4 GPU.**

In [ ]:
# 1. Unsloth telepítése (hivatalos Colab-installer — illeszti a torch/bnb/triton verziókat).
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --no-deps trl peft accelerate bitsandbytes

## Repo klónozása

A v4 tool-bővített dataset a **`main`-en** van (a v4 dataset PR mergelve). A notebook a main-t klónozza.
(Ha még merge ELŐTT futtatnád, írd át a lenti `-b main` kapcsolót a feature-branch nevére, pl.
`-b feature/benchmark-auto-judge`.)

In [ ]:
# 2. Repo a main-ről (a v4 tool-bővített dataset + finetune.py), majd be a training/-be.
!git clone --depth 1 -b main https://github.com/pits2022/free-droid.git
%cd free-droid/training
# Gyors ellenőrzés: a dataset a 681-es, tool-bővített legyen.
!python -c "import json; d=json.load(open('dataset/freedroid_full.json')); t=[x for x in d if '<tool>' in x['output']]; c=[x for x in t if x['output'].count('<tool>')>=2]; assert len(d)==681, f'VÁRT 681 példa (v4), de {len(d)} — a main még nem a v4 datasetet klónozta?'; print(f'peldak: {len(d)} | tool: {len(t)} ({100*len(t)/len(d):.1f}%) | osszetett: {len(c)}')"
!wc -l dataset/train.jsonl dataset/val.jsonl

## Edge modell — Llama 3.2 3B (offline fallback)

A Pi 5-ön, teljesen offline futó modell. Q4_K_M (edge) + Q8_0 (cloud) GGUF export.
Kimenet: `outputs/llama3.2-3b-v4/`.

In [ ]:
!python finetune.py --variant llama --preset gentle --tag v4

## Cloud modell — Llama 3.1 8B (a fő demó-agy, CPU-cloud)

A CAX31-en CPU-n futó nagyobb modell — a magyar/persona minőséget a méret hozza. A `llama8b` variáns
T4-biztos (batch=1, grad_accum=8) és csak Q4_K_M-et exportál. Kimenet: `outputs/llama3.1-8b-v4/`.

In [ ]:
!python finetune.py --variant llama8b --preset gentle --tag v4

## Next

- Kimenetek: `training/outputs/<variant>-v4/` — `gguf-q4_k_m` (edge), `gguf-q8_0` (cloud), `lora-adapter`.
  Töltsd le (vagy push HF Hubra) — git-ignorálva.
- **Ollama:** `ollama create szabi-3b-v4 -f Modelfile` / `szabi-8b-v4` (a `FROM` a kiválasztott GGUF-ra).
  Ügyelj a helyes `TEMPLATE`-re (Llama-3 fejléc) + stop tokenekre — az Unsloth GGUF NEM ágyazza be a sablont.
- **Benchmark:** `run_benchmark.py --models szabi-8b-v4 llama3.1:8b szabi-3b-v4 llama3.2:3b --rag` — a fő
  kérdés: **behozta-e a tool-bővítés a tool_calling-ot** (v3: 1–2/5), a persona/koherencia megtartásával.
  A `NUM_PREDICT=512` cap már benne van a runnerben.
- **Kötelező red-team pass** (provokatív / offtopic) a demó előtt.
- A tényeket futásidőben a `freedroid.rag` (offline BM25) adja, nem a fine-tune.